# Preprocessing with interpolation: forward-fill experiment

This notebook now uses **forward filling**, not linear interpolation. It repeats the most recent observed PM2.5 value on missing days and never uses a later reading to fill an earlier date. Leading blanks stay blank. Long gaps also repeat the last reading; these estimates are not actual measurements.

Run in VS Code with pandas and NumPy installed. Restart the kernel and run all cells in order. Inputs: five annual ZIP files in data/raw/. Outputs: separate original-POC-3 and corrected forward-filled CSVs in data/processed/. Existing no-fill CSVs are not overwritten. Preserve PM2.5_Original for scoring and the observed persistence baseline. Creating filled 2025 rows is preprocessing only, not model fitting or test evaluation.


In [1]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path


## Locate the project and the five input files


In [2]:
working_dir = Path.cwd().resolve()
project_root = next(
    (folder for folder in (working_dir, *working_dir.parents)
     if (folder / "data" / "raw").is_dir() and (folder / "scripts").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the notebook from the DS4002-project2 repository.")
files = [project_root / "data" / "raw" / f"daily_88101_{year}.zip"
         for year in range(2021, 2026)]
for filename in files:
    if not filename.is_file():
        raise FileNotFoundError(filename)
print("Project folder:", project_root)


Project folder: C:\Users\annie\OneDrive\Desktop\DS4002-project2


## Read the raw files, keeping only this site and hourly-summary rows

Filter each year before combining to reduce memory usage. Both POC versions are kept at this stage.


In [3]:
yearly_data = []
source_rows = 0
for filename in files:
    with zipfile.ZipFile(filename) as archive:
        csv_names = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        if len(csv_names) != 1:
            raise ValueError(f"Expected one CSV in {filename.name}")
        # Read in chunks so all nationwide data need not stay in memory.
        kept = []
        for chunk in pd.read_csv(archive.open(csv_names[0]), chunksize=100000, low_memory=False):
            source_rows += len(chunk)
            mask = (
                (chunk["State Code"] == 51) & (chunk["County Code"] == 3) &
                (chunk["Site Num"] == 1) & (chunk["Parameter Code"] == 88101) &
                (chunk["Sample Duration"] == "1 HOUR")
            )
            kept.append(chunk.loc[mask].copy())
        yearly_data.append(pd.concat(kept, ignore_index=True))
        print(filename.name, "site rows:", len(yearly_data[-1]), flush=True)
site_data = pd.concat(yearly_data, ignore_index=True)
site_data["Date Local"] = pd.to_datetime(site_data["Date Local"], errors="coerce")
for column in ["Arithmetic Mean", "Observation Count", "Observation Percent", "POC", "Method Code"]:
    site_data[column] = pd.to_numeric(site_data[column], errors="coerce")
print("Nationwide source rows read:", source_rows)


daily_88101_2021.zip site rows: 704
daily_88101_2022.zip site rows: 682
daily_88101_2023.zip site rows: 575
daily_88101_2024.zip site rows: 359
daily_88101_2025.zip site rows: 364
Nationwide source rows read: 4000854


## Select original POC 3 and corrected POC histories separately

Corrected history uses POC 23/method 736 before August 1, 2023 and POC 3/method 636 afterward. Both histories use the same quality and repeated-date rules.


In [4]:
cutoff = pd.Timestamp("2023-08-01")
original_mask = site_data["POC"] == 3
corrected_mask = (
    ((site_data["Date Local"] < cutoff) & (site_data["POC"] == 23) &
     (site_data["Method Code"] == 736)) |
    ((site_data["Date Local"] >= cutoff) & (site_data["POC"] == 3) &
     (site_data["Method Code"] == 636))
)
calendar = pd.date_range("2021-01-01", "2025-12-31", freq="D")


def clean_daily(selected):
    """Keep eligible observed summaries, then preserve the complete calendar."""
    selected = selected.dropna(subset=["Date Local", "Arithmetic Mean"]).copy()
    selected = selected.loc[selected["Observation Count"] >= 18].copy()
    selected["Event Priority"] = selected["Event Type"].map(
        {"Included": 2, "Excluded": 0}
    ).fillna(1)
    selected = selected.sort_values(
        ["Date Local", "Event Priority"], ascending=[True, False], kind="stable"
    ).drop_duplicates("Date Local", keep="first")
    selected = selected[[
        "Date Local", "Arithmetic Mean", "Observation Count", "Observation Percent",
        "Event Type", "Method Code", "Method Name"
    ]].rename(columns={
        "Date Local": "Date", "Arithmetic Mean": "PM2.5",
        "Observation Count": "Observation_Count", "Observation Percent": "Observation_Percent",
        "Event Type": "Event_Type", "Method Code": "Method_Code", "Method Name": "Method_Name"
    })
    return selected.set_index("Date").reindex(calendar).rename_axis("Date").reset_index()


observed_datasets = {
    "poc3": clean_daily(site_data.loc[original_mask]),
    "corrected_poc": clean_daily(site_data.loc[corrected_mask]),
}
for label, frame in observed_datasets.items():
    print(label, "calendar rows:", len(frame), "missing:", frame["PM2.5"].isna().sum())


poc3 calendar rows: 1826 missing: 67
corrected_poc calendar rows: 1826 missing: 67


## Verify the observed series against our existing no-fill exports

Stop if preprocessing differs before filling; do not silently compare different starting datasets.


In [ ]:
reference_names = {
    "poc3": "albemarle_pm25_poc3_without_interpolation.csv",
    "corrected_poc": "albemarle_pm25_corrected_poc_without_interpolation.csv",
}
for label, frame in observed_datasets.items():
    reference_path = project_root / "data" / "processed" / reference_names[label]
    if not reference_path.is_file():
        raise FileNotFoundError(f"Required no-fill reference missing: {reference_path}")
    reference = pd.read_csv(reference_path, parse_dates=["Date"])
    assert frame["Date"].equals(reference["Date"]), f"{label}: calendar differs"
    assert np.allclose(frame["PM2.5"], reference["PM2.5"], equal_nan=True), f"{label}: values differ"
    print(label, "matches its no-fill reference.")


## Forward fill: repeat the last observation, retain original values and flags

Do not backfill the leading gap. Leave coverage/method/event metadata blank on missing days rather than inventing measurement details.


In [6]:
filled_datasets = {}
for label, observed in observed_datasets.items():
    filled = observed.copy()
    filled["PM2.5_Original"] = filled["PM2.5"]
    filled["PM2.5"] = filled["PM2.5_Original"].ffill()
    filled["Was_Filled"] = filled["PM2.5_Original"].isna() & filled["PM2.5"].notna()
    # Track the actual observation date behind each supplied value.
    filled["Fill_Source_Date"] = filled["Date"].where(
        filled["PM2.5_Original"].notna()
    ).ffill()
    estimated = filled["Was_Filled"]
    observed_mask = filled["PM2.5_Original"].notna()
    assert np.allclose(filled.loc[observed_mask, "PM2.5"], filled.loc[observed_mask, "PM2.5_Original"])
    assert (filled.loc[estimated, "Fill_Source_Date"] < filled.loc[estimated, "Date"]).all()
    assert not filled["Date"].duplicated().any()
    filled_datasets[label] = filled
    print(label, "filled:", int(estimated.sum()),
          "still missing:", int(filled["PM2.5"].isna().sum()))
    print(filled.loc[estimated, ["Date", "PM2.5", "Fill_Source_Date"]].head().to_string(index=False))


poc3 filled: 61 still missing: 6
      Date    PM2.5 Fill_Source_Date
2021-05-03 6.108333       2021-05-02
2021-05-04 6.108333       2021-05-02
2021-05-05 6.108333       2021-05-02
2021-05-06 6.108333       2021-05-02
2021-05-07 6.108333       2021-05-02
corrected_poc filled: 61 still missing: 6
      Date    PM2.5 Fill_Source_Date
2021-05-03 5.204167       2021-05-02
2021-05-04 5.204167       2021-05-02
2021-05-05 5.204167       2021-05-02
2021-05-06 5.204167       2021-05-02
2021-05-07 5.204167       2021-05-02


## Save separate forward-filled CSVs

These are experimental model inputs. In training/evaluation, use PM2.5_Original for actual answers and the baseline, and PM2.5 for the model input. Compare on the same observed dates as the no-fill experiments.


In [7]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_names = {
    "poc3": "albemarle_pm25_poc3_forward_filled.csv",
    "corrected_poc": "albemarle_pm25_corrected_poc_forward_filled.csv",
}
for label, frame in filled_datasets.items():
    output_path = output_dir / output_names[label]
    frame.to_csv(output_path, index=False)
    print("Saved:", output_path)
print("No-fill CSVs were not overwritten. No models were trained or evaluated.")


Saved: C:\Users\annie\OneDrive\Desktop\DS4002-project2\data\processed\albemarle_pm25_poc3_forward_filled.csv
Saved: C:\Users\annie\OneDrive\Desktop\DS4002-project2\data\processed\albemarle_pm25_corrected_poc_forward_filled.csv
No-fill CSVs were not overwritten. No models were trained or evaluated.
